# Lab 4: Multi-Head Attention and Positional Encoding

**Course:** CS 417 — Large Language Models

**Name:** Muhammad Haseeb Ul Haq &nbsp;&nbsp;&nbsp; **Registration No.:** 454512


**Prerequisites:** Everything here builds directly on Lab 3. `stable_softmax` and `scaled_dot_product_attention` are given below exactly as you built them there, so you do not need to re-implement them. If Q/K/V, the attention formula, or the counterfactual-experiment idea feel shaky, it's worth a quick look back at Lab 3 before starting.

**Implementation rules:** As in Lab 3, do not use `torch.nn.MultiheadAttention` or any other prebuilt attention layer — the whole point of Task 5 is to build multi-head attention yourself, one piece at a time, out of parts you already trust.


In [ ]:
# Required packages: torch, matplotlib

import torch
import matplotlib.pyplot as plt

torch.manual_seed(0)


# Numerically stable softmax along `dim`. (Same as Lab 3, Task 3.)
def stable_softmax(x, dim=-1):
    m = x.max(dim=dim, keepdim=True).values
    e = torch.exp(x - m)
    return e / e.sum(dim=dim, keepdim=True)


# Single-head scaled dot-product attention. (Same as Lab 3, Task 3.)
#
# Q, K, V may have a leading dimension (batch, or attention heads) in
# addition to the usual [n, d] shape. Because this only ever operates on
# the LAST two dimensions, it broadcasts correctly over that leading
# dimension without any changes - this is exactly what makes it reusable,
# unmodified, for multi-head attention in Task 5.
def scaled_dot_product_attention(Q, K, V):
    d_k = Q.shape[-1]
    S = Q @ K.transpose(-2, -1) / (d_k ** 0.5)
    A = stable_softmax(S, dim=-1)
    Z = A @ V
    return Z, A, S


print("PyTorch version:", torch.__version__)


---
## Task 1 — The Bag-of-Words Problem: Self-Attention Has No Notion of Order

Self-attention decides how much each token attends to every other token using dot products between Query and Key vectors. Nothing in that computation looks at *where* a token sits in the sequence — only at its embedding.

This raises a question: if a query token's own embedding, and the **set** of words around it, both stay the same — but those surrounding words are simply reordered — does the query token's attention output change?

Below, `"bank"` sits at position 0 in both `context_1` and `context_2`; the other three words are the same set, just in a different order:

```text
context_1:  bank  loan  money  approved
context_2:  bank  approved  money  loan
```

**TODO:**

1. Implement `token_output(X, W_Q, W_K, W_V, token_index)`: project `X` into Q, K, V, run `scaled_dot_product_attention`, and return the output row at `token_index`.
2. Use it to compute `"bank"`'s contextualized output under `context_1` and under `context_2`, with the **same** projection matrices both times.
3. Compare the two outputs.


In [ ]:
E = {
    "bank":     torch.tensor([1.0, 0.5, 0.0, 0.0, 0.0, 0.0]),
    "loan":     torch.tensor([0.8, 0.3, 0.2, 0.0, 0.0, 0.0]),
    "money":    torch.tensor([0.7, 0.5, 0.3, 0.0, 0.0, 0.0]),
    "approved": torch.tensor([0.2, 0.1, 0.2, 0.1, 0.0, 0.0]),
}

context_1 = ["bank", "loan", "money", "approved"]
context_2 = ["bank", "approved", "money", "loan"]

X1 = torch.stack([E[w] for w in context_1])
X2 = torch.stack([E[w] for w in context_2])

torch.manual_seed(4)
W_Q = torch.randn(6, 4) * 0.3
W_K = torch.randn(6, 4) * 0.3
W_V = torch.randn(6, 4) * 0.3


def token_output(X, W_Q, W_K, W_V, token_index):
    Q = X @ W_Q
    K = X @ W_K
    V = X @ W_V
    Z, A, S = scaled_dot_product_attention(Q, K, V)
    return Z[token_index]


# --- test your implementation ---
bank_out_1 = token_output(X1, W_Q, W_K, W_V, token_index=0)
bank_out_2 = token_output(X2, W_Q, W_K, W_V, token_index=0)

print("bank output (context 1):", bank_out_1)
print("bank output (context 2):", bank_out_2)
print("identical despite reordered context:", torch.allclose(bank_out_1, bank_out_2, atol=1e-6))

assert torch.allclose(bank_out_1, bank_out_2, atol=1e-6), \
    "Without positional information, reordering the context should not change the output."


#### What I Did

I projected `X` into Q, K, V, ran `scaled_dot_product_attention`, and pulled out the row for `bank` (token 0). Running this on both context orderings with the same weights gives exactly the same output vector, confirming that self-attention alone cannot tell the two sentences apart.

**Analysis Questions**

1. `context_1` and `context_2` are different sentences — a human reading them would not treat them as identical. Why does self-attention treat `"bank"`'s situation as identical in both?
2. The formula $Z = AV$ is a weighted **sum**. Why does that make the result insensitive to the order in which the terms happen to be summed?
3. In Lab 3, two occurrences of the same word in one sentence got identical attention rows because they had no way to tell their positions apart. How is this task's finding the same underlying issue, seen from a different angle?
4. What real information could a language model lose if it genuinely could not tell "the dog bit the man" from "the man bit the dog"?


#### My Answers

1. Self-attention only ever looks at each token's embedding, through the Q/K dot products - nothing in that computation reads a token's position. Since the *set* of embeddings around bank and bank's own query are exactly the same numbers in both orderings, the scores, the softmax weights, and the weighted sum of values all come out identical either way. A human reads order as part of meaning; this computation never looks at order at all.

2. Addition is commutative: a + b + c gives the same total no matter what order you add the terms in. Z is literally a weighted sum of V vectors (`sum_j A_ij * V_j`), so as long as the same (weight, value) pairs are feeding into that sum, shuffling which one is "first" changes nothing about the total.

3. In Lab 3, two occurrences of the *same* word in one sentence got identical attention rows because they shared an embedding and had no position marker to tell them apart. Here it's the same root cause from the other side: now the *context* bank is attending to is identical as a set, just reordered, and since nothing marks order, bank's attention can't tell "this word came right after me" from "that word came three slots later." Same missing ingredient, different symptom.

4. The model would lose which noun is doing the action and which one is receiving it - exactly what makes these two sentences mean opposite things. With only a bag of words and no order, "the dog bit the man" and "the man bit the dog" look like the exact same set of words, so the model would have no way to know who bit whom.

---
## Task 2 — Sinusoidal Positional Encoding

Since self-attention alone cannot tell positions apart, the standard fix (used in the original Transformer paper) is to add a **position-dependent** vector to each token's embedding before running attention. For position $pos$ and dimension index $i$ (with model dimension $d_{model}$):

$$PE(pos, 2i) = \sin\!\left(\frac{pos}{10000^{2i/d_{model}}}\right), \qquad PE(pos, 2i+1) = \cos\!\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$

In words: even-numbered dimensions get a sine wave, odd-numbered dimensions get a cosine wave, and each pair of dimensions uses a different wavelength — low dimensions oscillate quickly (so adjacent positions look very different), high dimensions oscillate slowly. Every position ends up with its own fixed vector, and no two positions ever produce the same one.

**TODO:** Implement `positional_encoding(seq_len, d_model)`, returning a tensor of shape `[seq_len, d_model]`.


In [ ]:
def positional_encoding(seq_len, d_model):
    position = torch.arange(seq_len, dtype=torch.float32).unsqueeze(1)
    dim_index = torch.arange(d_model, dtype=torch.float32).unsqueeze(0)
    angle_rates = 1.0 / (10000 ** (2 * (dim_index // 2) / d_model))
    angles = position * angle_rates

    PE = torch.zeros(seq_len, d_model)
    PE[:, 0::2] = torch.sin(angles[:, 0::2])
    PE[:, 1::2] = torch.cos(angles[:, 1::2])
    return PE


# --- test your implementation ---
PE = positional_encoding(6, 8)
print("PE shape:", PE.shape)
print(PE)

assert PE.shape == (6, 8)
assert not torch.allclose(PE[0], PE[1]), "different positions must get different encodings"
assert torch.allclose(PE[0], torch.tensor([0., 1., 0., 1., 0., 1., 0., 1.]), atol=1e-6), \
    "position 0 should be [sin(0), cos(0), sin(0), cos(0), ...] = [0, 1, 0, 1, ...]"


#### What I Did

I built a `[seq_len, d_model]` grid of angles (position times a wavelength that depends on the dimension pair), then filled even columns with `sin` of that angle and odd columns with `cos` of the same angle. Position 0 comes out as `[0, 1, 0, 1, ...]` exactly as expected, since every angle is 0 at position 0.

In [ ]:
# A quick visual: each row is one position, each column one embedding
# dimension. Notice how low-index columns cycle quickly across rows while
# high-index columns barely change — that's the point: different dimensions
# encode position at different "resolutions."

plt.figure(figsize=(6, 3))
plt.imshow(PE.numpy(), aspect="auto", cmap="RdBu", vmin=-1, vmax=1)
plt.colorbar(label="value")
plt.xlabel("embedding dimension")
plt.ylabel("position")
plt.title("Positional encoding (6 positions, 8 dimensions)")
plt.show()


**Analysis Questions**

1. Why use a mix of many different sine/cosine wavelengths, instead of, say, just using the raw position number $pos$ itself as one extra feature?
2. Roughly why does using several different wavelengths together make it extremely unlikely that two different positions ever collide on the same encoding vector, even for long sequences?
3. Positional encodings are **added** to the token embedding ($X' = X + PE$) rather than kept as separate extra dimensions. What does addition assume about how "which word" and "which position" information can coexist in the same vector?


#### My Answers

1. A single raw number like `pos = 137` isn't on a useful scale for a network - it grows without bound as the sequence gets longer, and it would dominate the embedding once positions get into the hundreds. A sine/cosine value always stays in [-1, 1] no matter how long the sequence is, and layering many different wavelengths together gives a rich, bounded code for position instead of one unbounded scalar.

2. Each dimension pair cycles at its own wavelength, from very fast to very slow, and those wavelengths are spaced out exponentially (via the `10000^(2i/d_model)` term). For two different positions to land on the exact same full vector, every single one of those wavelengths would have to repeat at exactly the same time, and because they're spaced so far apart, their common repeat period is astronomically longer than any sequence anyone would actually use - like several clock hands moving at incompatible speeds, all lining up again at once.

3. Addition assumes "which word" and "which position" can be blended into the same set of numbers and still both be recoverable - that there's enough room in a `d_model`-sized vector for a mix of word meaning and position code without hopelessly destroying either one. It trades a clean separation (separate dimensions for word vs. position) for compactness, and it only works because the sine/cosine pattern is structured enough that attention's dot products can still pull the position signal back out.

---
## Task 3 — Positional Encoding Restores Order-Sensitivity

Repeat exactly the Task 1 experiment — same two context orderings, same projection matrices — but this time add positional encoding to the embeddings first: $X' = X + PE$.

**TODO:**

1. Implement `add_positions(X)`, returning `X + positional_encoding(X.shape[0], X.shape[1])`.
2. Recompute `"bank"`'s contextualized output (position 0) under both contexts, reusing `token_output` from Task 1.
3. Compare the two outputs — this time, they should differ.


In [ ]:
def add_positions(X):
    return X + positional_encoding(X.shape[0], X.shape[1])


# --- test your implementation ---
X1_pos = add_positions(X1)
X2_pos = add_positions(X2)

bank_out_1_pos = token_output(X1_pos, W_Q, W_K, W_V, token_index=0)
bank_out_2_pos = token_output(X2_pos, W_Q, W_K, W_V, token_index=0)

print("bank output with positions (context 1):", bank_out_1_pos)
print("bank output with positions (context 2):", bank_out_2_pos)
print("identical (should now be False):", torch.allclose(bank_out_1_pos, bank_out_2_pos, atol=1e-6))

assert not torch.allclose(bank_out_1_pos, bank_out_2_pos, atol=1e-6), \
    "With positional information, reordering the context should now change the output."

# Sanity check: bank's own position-0 input is still identical in both
# contexts (it's the same word at the same position both times) - so
# whatever changed the output must have come from the context, not from
# bank's own input vector.
assert torch.allclose(X1_pos[0], X2_pos[0], atol=1e-6)
print("bank's own input vector is still identical in both contexts:",
      torch.allclose(X1_pos[0], X2_pos[0], atol=1e-6))


#### What I Did

`add_positions` just adds the positional encoding to the embeddings before anything else runs. Repeating the exact same Task 1 experiment on `X + PE` now gives two different outputs for `bank`, even though `bank`'s own position-0 vector is still identical in both contexts — the only thing that changed is where the other words sit.

**Analysis Questions**

1. The code above confirms `bank`'s own embedding-plus-position vector is still identical in both contexts. If `bank`'s own input didn't change, what caused its *output* to change this time, when it didn't in Task 1?
2. Task 1 showed outputs stayed identical when only context order changed. Name precisely which quantity in the attention computation is now different between `context_1` and `context_2` that was not different before.
3. Would this experiment still detect a difference if, instead of reordering three distinct words, you swapped two *identical* repeated words between two positions? Why or why not?


#### My Answers

1. Bank's output is a weighted sum over the *other* tokens' keys and values too, not just its own. Even though bank's own embedding-plus-position didn't change, the other three words now each carry a different position vector depending on where they sit (`loan` at position 1 in context_1 vs. position 3 in context_2), so their keys and values are no longer the same between the two contexts - and that's what pulls the weighted sum in a different direction each time.

2. The attention scores `S` (and so the weights `A`) are now different between `context_1` and `context_2`, because the Keys being compared against bank's Query are `K = (word + position-specific PE) @ W_K`, and the position part of each surrounding word changed between the two orderings. In Task 1 every Key was identical across both orderings, so `S` and `A` came out identical; now they don't.

3. No - swapping two *identical* repeated words between two positions would not be detected, because the position-encoded vector for "whatever word sits at position 2" is the same regardless of which word is actually there. If the word at position 2 is the same both times, position 2's word-plus-PE vector is unchanged before and after the swap. The experiment only detects a difference when the actual content at each position changes, and swapping two copies of the same word leaves every position's content exactly the same.

---
## Task 4 — Splitting and Combining Attention Heads

So far, one query/key/value projection has to capture every kind of relationship between tokens at once — syntactic, positional, semantic — all mixed into a single score. **Multi-head attention** instead splits the model dimension into several smaller, independent groups (*heads*), lets each head compute its own attention pattern within that smaller subspace, and then puts the pieces back together.

If the model dimension is $d_{model}$ and there are $h$ heads, each head works with $d_{head} = d_{model} / h$ dimensions. Given a projected tensor of shape $[n, d_{model}]$ ($n$ tokens), splitting into heads produces a tensor of shape $[h, n, d_{head}]$ — the columns are cut into $h$ equal groups, and each group becomes its own "layer."

**TODO:**

1. Implement `split_heads(X, num_heads)`: reshape `[n, d_model]` into `[num_heads, n, d_head]`. Raise a `ValueError` if `d_model` is not evenly divisible by `num_heads`.
2. Implement `combine_heads(X)`: the exact inverse — reshape `[num_heads, n, d_head]` back into `[n, d_model]`.


In [ ]:
def split_heads(X, num_heads):
    n, d_model = X.shape
    if d_model % num_heads != 0:
        raise ValueError(f"d_model ({d_model}) must be evenly divisible by num_heads ({num_heads}).")
    d_head = d_model // num_heads
    return X.reshape(n, num_heads, d_head).permute(1, 0, 2)


def combine_heads(X):
    num_heads, n, d_head = X.shape
    return X.permute(1, 0, 2).reshape(n, num_heads * d_head)


# --- test your implementation ---
X_demo = torch.arange(24).float().reshape(4, 6)   # 4 tokens, d_model = 6

heads = split_heads(X_demo, num_heads=3)
print("split shape:", heads.shape)   # expect [3, 4, 2]

recombined = combine_heads(heads)
print("round-trip matches original:", torch.allclose(X_demo, recombined))

assert heads.shape == (3, 4, 2)
assert torch.allclose(X_demo, recombined)

try:
    split_heads(X_demo, num_heads=4)
    raised = False
except ValueError:
    raised = True
assert raised, "split_heads should reject a num_heads that doesn't evenly divide d_model"
print("correctly rejects an incompatible num_heads:", raised)


#### What I Did

`split_heads` cuts the columns into `num_heads` equal groups and moves the head axis to the front, so `[n, d_model]` becomes `[num_heads, n, d_head]`. `combine_heads` undoes exactly that same move. It checks divisibility first and raises a clear `ValueError` for a `num_heads` that doesn't evenly split `d_model`, like 4 heads over 6 dimensions.

**Analysis Questions**

1. If $d_{model} = 8$ and $num\_heads = 2$, what is $d_{head}$? Which specific columns of the original embedding end up in head 0, and which end up in head 1?
2. Why must `combine_heads` reverse the *exact* same reordering `split_heads` performed, rather than just reshaping the tensor back to `[n, d_model]` in whatever way happens to be convenient?
3. `split_heads` uses no learned weights at all — it is a pure reshape. Where in the multi-head pipeline (Task 5) does each head actually get the chance to specialize in something different from the others?


#### My Answers

1. `d_head = 8 / 2 = 4`. Head 0 gets the first 4 columns (indices 0-3) of the embedding, and head 1 gets the last 4 columns (indices 4-7) - `split_heads` just cuts the columns into `num_heads` equal, contiguous blocks.

2. `split_heads` moved the head axis to the front through one specific permutation, not just any reshape. If `combine_heads` used a *different* reshape that happened to produce the right final shape but mixed up which numbers land in which column, the values would be scrambled - one head's columns could blend into another's - giving a tensor of the right shape but nonsense content. Only reversing the exact same permutation guarantees the roundtrip recovers the original *values*, not just the original shape.

3. `split_heads` is pure bookkeeping - it never multiplies by anything learned. The actual specialization has to come from `W_Q`, `W_K`, and `W_V` themselves: since each head's slice of Q, K, V comes from a different block of columns in those projection matrices, gradient descent can push each block in a different direction during training, so each head's attention pattern can end up capturing something different. Splitting just creates the separate lanes; training decides what each lane learns to do.

---
## Task 5 — Assembling Multi-Head Self-Attention

With `split_heads` / `combine_heads` from Task 4, and `scaled_dot_product_attention` from Lab 3, you have everything needed to assemble full multi-head attention:

$$Q = XW_Q, \quad K = XW_K, \quad V = XW_V \qquad \text{(all at full } d_{model}\text{)}$$

$$Q_h, K_h, V_h = \text{split\_heads}(Q),\ \text{split\_heads}(K),\ \text{split\_heads}(V)$$

$$Z_h, A_h, S_h = \text{scaled\_dot\_product\_attention}(Q_h, K_h, V_h) \qquad \text{(runs independently, per head)}$$

$$\text{output} = \text{combine\_heads}(Z_h)\, W_O$$

Notice that `scaled_dot_product_attention` does not need to change at all — because it only operates on the *last two* dimensions of whatever you give it, it automatically runs once per head when handed a `[num_heads, n, d_head]` tensor. $W_O$ is one final learned projection that mixes information back across heads after they have each attended independently.

**TODO:** Implement `multi_head_attention(X, W_Q, W_K, W_V, W_O, num_heads)`, returning the final output tensor of shape `[n, d_model]` and the per-head attention weights `A_h` of shape `[num_heads, n, n]`.


In [ ]:
d_model = 8
num_heads = 2
n_tokens = 5

torch.manual_seed(7)
X_mha = torch.randn(n_tokens, d_model)
W_Q5 = torch.randn(d_model, d_model) * 0.3
W_K5 = torch.randn(d_model, d_model) * 0.3
W_V5 = torch.randn(d_model, d_model) * 0.3
W_O5 = torch.randn(d_model, d_model) * 0.3


def multi_head_attention(X, W_Q, W_K, W_V, W_O, num_heads):
    Q = X @ W_Q
    K = X @ W_K
    V = X @ W_V

    Q_h = split_heads(Q, num_heads)
    K_h = split_heads(K, num_heads)
    V_h = split_heads(V, num_heads)

    Z_h, A_h, S_h = scaled_dot_product_attention(Q_h, K_h, V_h)

    output = combine_heads(Z_h) @ W_O
    return output, A_h


# --- test your implementation ---
output, A_h = multi_head_attention(X_mha, W_Q5, W_K5, W_V5, W_O5, num_heads)
print("output shape:", output.shape)      # expect [5, 8]
print("attention shape:", A_h.shape)      # expect [2, 5, 5]

assert output.shape == (n_tokens, d_model)
assert A_h.shape == (num_heads, n_tokens, n_tokens)
assert torch.allclose(A_h.sum(dim=-1), torch.ones(num_heads, n_tokens), atol=1e-5), \
    "each attention row, in every head, must sum to 1"


# Cross-check: an independent per-head loop, with no reshape tricks at all,
# must produce exactly the same answer as your reshape-based version.
def multi_head_attention_loop(X, W_Q, W_K, W_V, W_O, num_heads):
    n, d_model = X.shape
    d_head = d_model // num_heads
    Q, K, V = X @ W_Q, X @ W_K, X @ W_V
    outs = []
    for h in range(num_heads):
        sl = slice(h * d_head, (h + 1) * d_head)
        Zh, _, _ = scaled_dot_product_attention(Q[:, sl], K[:, sl], V[:, sl])
        outs.append(Zh)
    Z = torch.cat(outs, dim=-1)
    return Z @ W_O


output_loop = multi_head_attention_loop(X_mha, W_Q5, W_K5, W_V5, W_O5, num_heads)
print("matches independent per-head loop:", torch.allclose(output, output_loop, atol=1e-5))
assert torch.allclose(output, output_loop, atol=1e-5), \
    "your reshape-based implementation must match an independent per-head loop"


#### What I Did

I projected `X` into full-size Q, K, V, split each into heads, ran the same `scaled_dot_product_attention` from Lab 3 unchanged (it just broadcasts over the head axis), combined the heads back together, and applied the final `W_O` mix. It matches an independently written per-head loop exactly, so the reshape-based version is doing the same thing, just without a Python loop.

**Analysis Questions**

1. Why does `scaled_dot_product_attention` work correctly on a `[num_heads, n, d_head]` input without any code changes at all? What property of matrix multiplication and softmax makes that possible?
2. If you set $h = 1$ (one head), `multi_head_attention` should reduce to (almost) exactly single-head attention from Lab 3, aside from the extra $W_O$ at the end. Why might it still be worth keeping $W_O$ even when there is only one head?
3. Each head only ever sees $d_{head}$ dimensions instead of the full $d_{model}$. Why might having several heads, each working in a smaller subspace, still let the model capture *more* useful patterns overall than one head using the full $d_{model}$?


#### My Answers

1. Matrix multiplication and softmax, as PyTorch implements them, only operate on the *last two* dimensions of whatever tensor they're given, treating any leading dimensions as independent batches processed in parallel. A `[num_heads, n, d_head]` tensor is exactly a batch of `num_heads` separate `[n, d_head]` matrices stacked together, so the same code that worked on one `[n, d_head]` matrix in Lab 3 automatically repeats itself once per head - nothing about the function needed to know heads existed.

2. Even with one head, `W_O` still gives the model one more learned linear transformation to apply after attention, letting it remix or rescale the output into whatever shape is most useful for the next layer, instead of being stuck with attention's raw output directly. It costs very little and keeps the function's interface identical whether `num_heads` is 1 or many, so it works as a drop-in replacement for single-head attention without special-casing.

3. A single head computing one big dot product over all `d_model` dimensions is forced to blend every kind of relationship - syntax, position, topic - into one shared similarity score, so two relationships that would pull the score in opposite directions can just cancel out. Splitting into several smaller heads lets each one specialize in its own narrower pattern without interference from the others, and `W_O` recombines those specialized views afterward - trading one broad, compromise-prone view for several focused ones that can complement each other.

---
## Task 6 — Comparing What Different Heads Attend To

Just as Lab 3 treated a single attention matrix as data rather than a picture, this task compares **multiple heads' attention matrices**, computed on the same sentence, to see whether they actually differ from one another.

Recall from Lab 3: the entropy of one attention row measures how spread out (high entropy) or concentrated (low entropy) that row's attention is:

$$H(A_i) = -\sum_j a_{ij}\log(a_{ij})$$

**TODO:**

1. Implement `head_entropy(A_head)`: given one head's attention matrix (`[n, n]`), return a length-`n` tensor of each row's entropy.
2. Implement `most_different_heads(A_h)`: given all heads' attention (`[num_heads, n, n]`), find the pair of heads whose attention matrices differ the most — using the total absolute difference, summed over all entries, as the comparison — and return `(i, j, difference_value)` for that pair.


In [ ]:
def head_entropy(A_head, eps=1e-12):
    return -(A_head * torch.log(A_head + eps)).sum(dim=-1)


def most_different_heads(A_h):
    num_heads = A_h.shape[0]
    best = None
    best_diff = -1.0
    for i in range(num_heads):
        for j in range(i + 1, num_heads):
            diff = float((A_h[i] - A_h[j]).abs().sum())
            if diff > best_diff:
                best_diff = diff
                best = (i, j, diff)
    return best


# --- test your implementation ---
entropies = torch.stack([head_entropy(A_h[h]) for h in range(num_heads)])
print("entropy per head, per token (rows = heads, columns = tokens):")
print(entropies)

assert entropies.shape == (num_heads, n_tokens)
assert torch.all(entropies >= 0), "entropy cannot be negative"

i, j, diff = most_different_heads(A_h)
print(f"\nmost different pair of heads: ({i}, {j}), total abs. difference = {diff:.4f}")


#### What I Did

`head_entropy` applies the same entropy formula from Lab 3 to every row of one head's attention matrix. `most_different_heads` checks every pair of heads and returns the one whose attention matrices differ the most, measured as the total absolute difference across every entry.

**Analysis Questions**

1. Do the different heads' entropy values for the *same* token differ noticeably? What would it mean, in plain terms, if head 0 has low entropy on a token while head 1 has high entropy on that same token?
2. These heads were initialized with random weights and never trained. What would you expect to happen to how "different" the heads are from one another after training on real language data — and why?
3. In Task 5's third analysis question, you speculated about heads specializing in different kinds of relationships. Based on what you actually found here, do these (untrained, random) heads show any real sign of that kind of specialization? Why should you be cautious about reading too much into differences between randomly initialized heads?


#### My Answers

1. Somewhat, yes - at token index 2, head 0's entropy is 1.2680 while head 1's is 1.5752, a noticeably bigger gap than at most of the other tokens. In plain terms, that would mean head 0 is concentrating its attention on a small number of tokens at that position (more "decisive"), while head 1 is spreading its attention more evenly across most of the tokens for that same position (more "unsure" or diffuse) - two heads looking at the same token but coming away with differently shaped attention.

2. I would expect the gap between heads to generally grow larger and more consistent after training on real data, because gradient descent would be actively pushing each head's `W_Q`/`W_K` toward whatever pattern reduces the training loss, and if two heads end up useful for different reasons, the loss would tend to reward them becoming more distinct rather than redundant. Right now, with random untrained weights, any difference between heads is just a side effect of the random initialization, not something the network has decided is useful.

3. Not really. With only 2 heads, the "most different pair" is trivially (0, 1) - there is no other pair to compare it against - and the actual entropy values are all clustered fairly close together, roughly 1.27 to 1.59 nats, with a total pairwise difference of about 2.04. That is not strong evidence of real specialization; it looks more like the modest variation you would expect from two random matrices by chance. The caution is that it is tempting to look at any numeric gap and tell a story about what each head "learned to do," but with untrained, randomly initialized weights, there is nothing there to have learned anything yet - the difference just reflects the specific random seed, not any discovered linguistic pattern.

---
## Deliverable

Submit this notebook with your name and registration number clearly written at the top, all `TODO` cells completed with their test/validation cells executed successfully, and concise written answers to every task's analysis questions.
